# Edith Operational Notebook

This notebook is a thin, simulation-only operator and research interface for the current Lilith repository. It imports tested Python modules instead of duplicating trading logic.

**Safety boundary:** this notebook does not connect to MetaTrader 5, submit orders, or enable live execution.

In [1]:
import importlib.util
import os
import sys
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()
SRC_ROOT = REPOSITORY_ROOT / 'src'
if SRC_ROOT.exists() and str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

EXECUTION_MODE = os.getenv('LILITH_EXECUTION_MODE', 'simulation').strip().lower()
if EXECUTION_MODE != 'simulation':
    raise RuntimeError('This notebook is simulation-only. Set LILITH_EXECUTION_MODE=simulation.')

required_modules = ('lilith', 'pandas')
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        'Missing required modules: ' + ', '.join(missing) +
        '. Install the project with: python -m pip install -e .'
    )

print(f'Repository: {REPOSITORY_ROOT}')
print(f'Execution mode: {EXECUTION_MODE}')
print('Environment check: PASS')

Repository: C:\Users\Chaba\Documents\Lilith
Execution mode: simulation
Environment check: PASS


## Trade-forensics smoke test

This deterministic simulation exercises the same public interfaces covered by the repository tests.

In [2]:
from datetime import datetime, timedelta, timezone
from decimal import Decimal

from lilith.forensics.models import EntrySnapshot, LifecycleSnapshot, RealisedOutcome, Side
from lilith.forensics.service import TradeForensicsService

start = datetime(2026, 7, 28, 8, 0, tzinfo=timezone.utc)
entry = EntrySnapshot(
    trade_id='NB-T-1', signal_id='NB-S-1', symbol='XAUUSDm', timeframe='M5',
    side=Side.BUY, timestamp=start, requested_entry=Decimal('100'),
    filled_entry=Decimal('100'), stop_price=Decimal('99'),
    target_price=Decimal('102'), volume=Decimal('0.01'),
    balance=Decimal('50'), equity=Decimal('50'),
)
snapshots = [
    LifecycleSnapshot('NB-T-1', start, Decimal('100.2'), Decimal('100.3'), floating_pnl=Decimal('0.2')),
    LifecycleSnapshot('NB-T-1', start + timedelta(minutes=1), Decimal('101.2'), Decimal('101.3'), floating_pnl=Decimal('1.2')),
    LifecycleSnapshot('NB-T-1', start + timedelta(minutes=2), Decimal('100.4'), Decimal('100.5'), floating_pnl=Decimal('0.4')),
    LifecycleSnapshot('NB-T-1', start + timedelta(minutes=3), Decimal('99'), Decimal('99.1'), floating_pnl=Decimal('-1')),
]
outcome = RealisedOutcome(
    trade_id='NB-T-1', exit_timestamp=start + timedelta(minutes=3),
    exit_price=Decimal('99'), gross_profit=Decimal('-1'), broker_reason='SL',
)
report = TradeForensicsService().analyse(entry, snapshots, outcome)
assert report.primary_cause == 'profitable excursion was not protected'
assert report.net_realised_pnl == Decimal('-1')
print('Trade forensics smoke test: PASS')
print({
    'exit_reason': report.exit_reason.value,
    'mfe_r': str(report.mfe_r),
    'mae_r': str(report.mae_r),
    'management_quality': report.management_quality,
    'primary_cause': report.primary_cause,
})

Trade forensics smoke test: PASS
{'exit_reason': 'STOP_LOSS', 'mfe_r': '1.2', 'mae_r': '1', 'management_quality': 'breakeven_or_trailing_candidate', 'primary_cause': 'profitable excursion was not protected'}


## Feature-sculpting and sizing smoke tests

These cells use generated observations only. They do not read broker data or alter strategy behaviour.

In [3]:
from lilith.sculpting import FeatureSculptor, SculptorPolicy, TradeObservation
from lilith.sizing import PositionSizingResearch, SizingObservation

observations = [
    TradeObservation(
        trade_id=str(index),
        net_pnl=Decimal('2') if index % 4 else Decimal('-1'),
        r_multiple=Decimal('0.5') if index % 4 else Decimal('-0.25'),
        features={'session': 'NEW_YORK', 'regime': 'TREND'},
    )
    for index in range(40)
]
policy = SculptorPolicy(
    minimum_sample=30, minimum_expectancy_r=Decimal('0.05'),
    minimum_profit_factor=Decimal('1.10'), maximum_drawdown=Decimal('10'),
    minimum_stability_score=0.60,
)
sculpting_results = FeatureSculptor(policy).analyse(observations)
target = next(item for item in sculpting_results if item.fingerprint == 'regime=TREND|session=NEW_YORK')
assert target.approved is True

sizing_rows = [
    SizingObservation(
        str(index), Decimal('1') if index % 2 else Decimal('-0.5'),
        Decimal('0.8'), Decimal('1.2'),
    )
    for index in range(20)
]
sizing_results = PositionSizingResearch().compare(sizing_rows, starting_equity=Decimal('100'))
assert {item.policy for item in sizing_results} == {'fixed_risk', 'confidence_scaled', 'volatility_scaled'}
print('Feature sculpting smoke test: PASS')
print('Position sizing research smoke test: PASS')
print({
    'fingerprint': target.fingerprint,
    'approved': target.approved,
    'sample_size': target.sample_size,
    'expectancy_r': str(target.expectancy_r),
    'profit_factor': str(target.profit_factor),
})

Feature sculpting smoke test: PASS
Position sizing research smoke test: PASS
{'fingerprint': 'regime=TREND|session=NEW_YORK', 'approved': True, 'sample_size': 40, 'expectancy_r': '0.3125', 'profit_factor': '6'}


## Operator guidance

- Install the repository with `python -m pip install -e .`.
- Keep `LILITH_EXECUTION_MODE=simulation` when using this notebook.
- Run the repository test suite with `python -m pytest -q`.
- Use the Streamlit dashboard entrypoint defined by the repository rather than embedding dashboard code here.
- MetaTrader 5 connectivity and live execution are intentionally excluded from this notebook.

## MT5 Demo session

Connects to a MetaTrader 5 **demo** account using `lilith.mt5_demo.MT5DemoRuntime`.

**Safety boundary:** the runtime enforces demo-only mode — it will refuse to run against a live account.  
Run the cell below to connect, poll one market tick, and print account telemetry. No order is placed by these cells; to start the full trading loop use `run_from_environment()` in a script instead.

In [4]:
import os

# ── Demo session configuration ────────────────────────────────────────────────
# ⚠ Remove credentials before sharing or committing this notebook to git.
_demo_env = {
    'LILITH_EXECUTION_MODE': 'mt5-demo',
    'EDITH_MT5_CONFIRM_DEMO': 'YES',
    'MT5_LOGIN': '436965532',
    'MT5_PASSWORD': 'killer$Am3',
    'MT5_SERVER': 'Exness-MT5Trial9',
    'EDITH_MT5_SYMBOL': 'XAUUSDm',
    'EDITH_MT5_TIMEFRAME': 'M5',
    'EDITH_MT5_LOT': '0.01',
    'EDITH_MT5_POLL_SECONDS': '15',
    'EDITH_MT5_MAX_POSITIONS': '1',
}
for _k, _v in _demo_env.items():
    os.environ.setdefault(_k, _v)

from lilith.mt5_demo import MT5DemoRuntime

runtime = MT5DemoRuntime()
runtime.connect()

_mt5 = runtime.mt5
_account = _mt5.account_info()
print('MT5 Demo connection: PASS')
print(f'  Account  : {_account.login}')
print(f'  Server   : {_account.server}')
print(f'  Balance  : {_account.balance} {_account.currency}')
print(f'  Equity   : {_account.equity} {_account.currency}')
print(f'  Profit   : {_account.profit} {_account.currency}')
print(f'  Mode     : {"DEMO" if int(_account.trade_mode) == 0 else "LIVE — ABORTING"}')
if int(_account.trade_mode) != 0:
    _mt5.shutdown()
    raise RuntimeError('Account is not a demo account. Aborting.')

MT5 Demo connection: PASS
  Account  : 436965532
  Server   : Exness-MT5Trial9
  Balance  : 11.07 USD
  Equity   : 11.07 USD
  Profit   : 0.0 USD
  Mode     : DEMO


In [5]:
# ── Single market poll (no order placed) ─────────────────────────────────────
fast, slow, atr = runtime.market()
sig = runtime.signal(fast, slow, atr)

print('Market poll:')
for _k, _v in sig.items():
    print(f'  {_k}: {_v}')

_mt5.shutdown()
print('\nMT5 shutdown: OK')

Market poll:
  timestamp: 2026-07-29T19:53:43.575388+00:00
  session_id: 7a165f1d-34fe-4c4f-bfbf-744c14b40622
  iteration: 0
  symbol: XAUUSDm
  timeframe: M5
  signal: SELL
  decision: ENTER_MT5_DEMO
  score: 100.0
  fast_sma: 4069.7584
  slow_sma: 4082.4431
  atr: 11.88807
  mode: mt5-demo
  reason: fast/slow SMA direction with ATR-normalised confidence

MT5 shutdown: OK
